<a href="https://colab.research.google.com/github/Krish-8957/IPL-WIN-PREDICTOR/blob/main/ipl_win_probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

matches = pd.read_csv("/content/matches (1).csv")
deliveries = pd.read_csv("/content/deliveries.csv (1).zip")

print(matches.columns.tolist())
print(deliveries.columns.tolist())

['id', 'Season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs', 'win_by_wickets', 'player_of_match', 'venue', 'umpire1', 'umpire2', 'umpire3']
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batsman', 'non_striker', 'bowler', 'is_super_over', 'wide_runs', 'bye_runs', 'legbye_runs', 'noball_runs', 'penalty_runs', 'batsman_runs', 'extra_runs', 'total_runs', 'player_dismissed', 'dismissal_kind', 'fielder']


In [2]:
# First innings total
total_score_df = deliveries.groupby(['match_id', 'inning'])['total_runs'].sum().reset_index()

total_score_df = total_score_df[total_score_df['inning'] == 1]

# Merge target with matches
match_df = matches.merge(
    total_score_df[['match_id', 'total_runs']],
    left_on='id',
    right_on='match_id'
)

teams = [
    'Sunrisers Hyderabad',
    'Mumbai Indians',
    'Royal Challengers Bangalore',
    'Kolkata Knight Riders',
    'Kings XI Punjab',
    'Chennai Super Kings',
    'Rajasthan Royals',
    'Delhi Daredevils'
]

match_df = match_df[match_df['team1'].isin(teams)]
match_df = match_df[match_df['team2'].isin(teams)]

match_df = match_df[match_df['dl_applied'] == 0]

match_df = match_df[['id', 'city', 'winner', 'total_runs']]

# Correct merge
delivery_df = match_df.merge(
    deliveries,
    left_on='id',
    right_on='match_id'
)

delivery_df = delivery_df[delivery_df['inning'] == 2]

# Current score
delivery_df['current_score'] = delivery_df.groupby('match_id')['total_runs_y'].cumsum()

# Runs left
delivery_df['runs_left'] = delivery_df['total_runs_x'] - delivery_df['current_score'] + 1

# Balls left
delivery_df['balls_left'] = 126 - (
    delivery_df['over'] * 6 + delivery_df['ball']
)

# Wickets left
delivery_df['player_dismissed'] = delivery_df['player_dismissed'].fillna("0")

delivery_df['player_dismissed'] = delivery_df['player_dismissed'].apply(
    lambda x: 0 if x == "0" else 1
)

wickets = delivery_df.groupby('match_id')['player_dismissed'].cumsum()

delivery_df['wickets_left'] = 10 - wickets

# Current Run Rate
delivery_df['crr'] = (
    delivery_df['current_score'] * 6
) / (120 - delivery_df['balls_left'])

# Required Run Rate
delivery_df['rrr'] = (
    delivery_df['runs_left'] * 6
) / delivery_df['balls_left']

# Result column
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0

delivery_df['result'] = delivery_df.apply(result, axis=1)

final_df = delivery_df[
    [
        'batting_team',
        'bowling_team',
        'city',
        'runs_left',
        'balls_left',
        'wickets_left',
        'total_runs_x',
        'crr',
        'rrr',
        'result'
    ]
]

final_df = final_df.sample(final_df.shape[0])

final_df.dropna(inplace=True)

final_df = final_df[final_df['balls_left'] != 0]

X = final_df.iloc[:, :-1]
y = final_df.iloc[:, -1]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=1
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

trf = ColumnTransformer(
    transformers=[
        (
            'trf',
            OneHotEncoder(handle_unknown='ignore'),
            ['batting_team', 'bowling_team', 'city']
        )
    ],
    remainder='passthrough'
)

pipe = Pipeline([
    ('step1', trf),
    ('step2', LogisticRegression(max_iter=1000))
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

print("Accuracy =", accuracy_score(y_test, y_pred))



Accuracy = 0.8071148825065274


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [3]:
import pickle

pickle.dump(pipe, open('pipe.pkl', 'wb'))

print("pipe.pkl saved successfully")

pipe.pkl saved successfully


In [ ]:
import sys
import sklearn
import pandas
import numpy

print("Python:", sys.version)
print("Scikit-Learn:", sklearn.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Scikit-Learn: 1.6.1
Pandas: 2.2.2
NumPy: 2.0.2
